# benchmark multiple openai models using repetitive sums dataset and few token completion method  

We give the chat completion api a chat hisotry that contains part of the assistant response and only ask for a few tokens back.  

From there, we extract the response.  

## install libs

In [ ]:
!pip install datasets openai

## inference function  

openai models can answer from 2 to 100 using a single token.  

### define inference function  



In [ ]:
from openai import OpenAI
from google.colab import userdata
import time

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

def extract_number(input_str):
  try:
    return int(''.join(filter(str.isdigit, input_str)))
  except ValueError:
    return 0

extract_number("bla bla . bla bla"), extract_number("bla bla 22 bla bla")

def get_sum_result(current_sum, curr_model):
  while True:
    try:
      response = client.chat.completions.create(
        model=curr_model,
        messages=[
          {"role": "system", "content": "You are a helpful assistant. Answer with a single token."},
          {"role": "user", "content": f"What is the result of the following sum : {current_sum} ?"},
        ],
        max_tokens = 1
      )
      return extract_number(response.choices[0].message.content )
    except Exception: #catch http 429 here
      print("Retrying...")
      time.sleep(60)

### test inference function

In [ ]:
test_sums = [
  '1+1',
  '2+2',
  '50+50'
]

models_to_test = [
    "gpt-4o-2024-05-13",
    "gpt-4-turbo-2024-04-09",
    "gpt-4-0125-preview",
    "gpt-4-1106-preview",
    "gpt-4-0613",
    "gpt-3.5-turbo-0125",
    "gpt-3.5-turbo-1106",
]

for curr_model in models_to_test:
  print(f"--- Testing model : {curr_model} ---")
  for curr_sum in test_sums:
    llm_result = get_sum_result(curr_sum, curr_model)
    is_success = llm_result  == eval(curr_sum)
    if is_success:
      print(f"✅ {curr_sum} = {llm_result}")
    else:
      print(f"❌ {curr_sum} = {llm_result}")



## define benchmark functions

In [ ]:
from functools import partial
import concurrent.futures
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd


def perform_parallel_sums(sums_to_compute, model_name):
    # Number of parallel calls
    N = 4

    # Create a partial function with model_name argument
    partial_get_sum_result = partial(get_sum_result, curr_model=model_name)

    # Create a ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor(max_workers=N) as executor:
        # Wrap the executor.map with tqdm for progress bar
        inferred_sums = list(tqdm(executor.map(partial_get_sum_result, sums_to_compute), total=len(sums_to_compute)))

    return inferred_sums

def get_benchmark_df(df, models_to_test, number_of_samples=10):

  inferred_sums = []
  for curr_model in tqdm(models_to_test, desc='inferring on models'):
      inferred_sums.append(perform_parallel_sums(df['sum'].tolist()[:number_of_samples], curr_model))

  df_list = []
  df_list.append(df.head(number_of_samples))

  for inference_results, model_name in zip(inferred_sums, models_to_test):
    df_list.append(pd.DataFrame(inference_results, columns=[model_name]))
  return pd.concat(df_list, axis=1)

def analyze_model(df, model_name):
  df = df.copy(deep=True)

  # count parsing failures
  number_of_parsing_failures = len(df[df[model_name] == 0])

  # compute accuracy including JSON parse errors as bad answers
  df['is_correct'] = df['result'].astype(int) == df[model_name]
  mean_accuracy = df.is_correct.mean()*100

  # exclude parsing failures from stat count as "0" is not a significant result to measure
  df = df[df[model_name] != 0]

  # select the failed results to compute error stats
  wrong_answers = df[df.is_correct == False].copy(deep=True)


  if len(wrong_answers) == 0:
    return {
        'model_name' : model_name,
        'avg_accuracy' : mean_accuracy,
        'error_mean' : 0,
        'error_median' : 0,
        'error_std' : 0,
        'error_min' : 0,
        'error_max' : 0,
        'parsing_failure_count' : number_of_parsing_failures
    }
  else:
    wrong_answers['error_abs'] = abs(wrong_answers['result'].astype(int) - wrong_answers[model_name])
    wrong_answers['error_prop'] = wrong_answers['error_abs'] / wrong_answers['result'].astype(int)
    return {
        'model_name' : model_name,
        'avg_accuracy' : mean_accuracy,
        'error_mean' : wrong_answers['error_abs'].mean(),
        'error_median' : wrong_answers['error_abs'].median(),
        'error_std' : wrong_answers['error_abs'].std(),
        'error_min' : wrong_answers['error_abs'].min(),
        'error_max' : wrong_answers['error_abs'].max(),
        'parsing_failure_count' : number_of_parsing_failures
    }

def get_stats_recap(benchmark_df, models_to_test):
  results = []
  for model_name in models_to_test:
    results.append(analyze_model(benchmark_df, model_name))

  return pd.DataFrame(results)

def show_model_results(df, model_name):
  df['is_correct'] = df['result'].astype(int) == df[model_name]

  sns.set_theme(style="whitegrid")

  # Order the results
  results = df['result'].astype(int).unique()
  results.sort()

  # set the high and width
  plt.figure(figsize=(20, 2))

  # Create the barplot
  g = sns.barplot(x="result", y=[1]*len(df), data=df, order=results, hue="is_correct", palette={False:"#EA5455", True:"#34A853"})

  # pivot label
  g.set_xticklabels(g.get_xticklabels(), rotation=90, ha="right")

  # Add title and labels
  plt.title(f"Model : {model_name} - Position of correct sums")
  plt.xlabel("Sum")
  plt.show()


## Load dataset

In [ ]:
from datasets import load_dataset

dataset_ds = load_dataset("the-french-artist/repetitive_sums_benchmark", split='train')
dataset_ds

In [ ]:
additions_df = dataset_ds.to_pandas()

## Perform inference on complete dataset  

In [ ]:
models_to_test = [
    "gpt-4o-2024-05-13",
    "gpt-4-turbo-2024-04-09",
    "gpt-4-0125-preview",
    "gpt-4-1106-preview",
    "gpt-4-0613",
    "gpt-3.5-turbo-0125",
    "gpt-3.5-turbo-1106",
]
number_of_samples = 100

benchmark_df = get_benchmark_df(additions_df, models_to_test, number_of_samples)
benchmark_df.head()

## Analyze results

In [ ]:
get_stats_recap(benchmark_df, models_to_test)

In [ ]:
for curr_model in models_to_test:
  show_model_results(benchmark_df, curr_model)

## update the leaderboard

In [ ]:
from datasets import load_dataset

leaderboard_ds = load_dataset('the-french-artist/repetitive_sums_benchmark_leaderboard', split='train')
leaderboard_ds

In [ ]:
leaderboard_df = leaderboard_ds.to_pandas()
leaderboard_df

In [ ]:
leaderboard_df_update = get_stats_recap(benchmark_df, models_to_test)
leaderboard_df_update

In [ ]:
import pandas as pd

updated_leaderboard = pd.concat([leaderboard_df, leaderboard_df_update])
updated_leaderboard = updated_leaderboard.sort_values('avg_accuracy', ascending=False)
updated_leaderboard

In [ ]:
from datasets import Dataset
updated_leaderboard_ds = Dataset.from_pandas(updated_leaderboard, preserve_index=False)
updated_leaderboard_ds

In [ ]:
from google.colab import userdata

updated_leaderboard_ds.push_to_hub('the-french-artist/repetitive_sums_benchmark_leaderboard', token=userdata.get('HF_TOKEN'))